# 07 · Doctor Strange's time stone
*"Which version of this record was true last Tuesday?"*

Hydra got into the incident table and quietly downgraded every `critical` incident to `low`. Delta kept every version, so you can ask who did it,
which rows changed, what the table looked like before, and then put it back. You tamper with the table yourself so you can watch the whole story.

| Concept | Where you meet it |
|---|---|
| Table history: every commit is a version you can list | the next section, then Step 3 |
| Change data feed (CDF): row-level before and after images of each change | the next section, then Steps 1 and 4 |
| Time travel: query an older version | the next section, then Step 5 |
| `RESTORE`: make the table look like an older version again | the next section, then Steps 6 and 7 |

Deep dive: [docs/07-doctor-strange-time-stone.md](../../docs/07-doctor-strange-time-stone.md)

**Needs:** `shield.silver.incident_reports` with some `critical` incidents. Chapter 06 builds it (9,500 incidents); chapter 05 or 10 also make a version of it. Any compute works, serverless included, and you need `MODIFY` on the table.
If the table is missing, or has no `critical` rows, the setup cell stops with a message; fix it by running chapter 06 (it rebuilds the table).
The **Expected** numbers assume the table as chapter 06 leaves it, built from the dataset of chapter 02 (seed 42, 5 % dirty); after chapter 05 or 10 the counts differ and the check tables will show ❌, which is not a bug.
Nothing here has been run in a workspace yet, so every **Expected** is an expectation, not a recorded result.

**Side effects.** This notebook changes a real table: it turns the change feed on, updates 435 rows, and restores them. The data ends up as it started, but the history gains a few versions that never go away, and the table property is switched off again in the clean-up.
Run the steps in order. If you stop between Step 2 and Step 6 the table stays tampered: finish with Step 6, or rebuild the table by re-running chapter 06.

## Table history, time travel, RESTORE and the change data feed

**The idea:** in chapter 04 you saw that every write to a Delta table is a numbered commit. Four tools read or use those versions. They are Delta features, not patterns.

```
 version   operation                                     what the table holds
 ───────   ───────────────────────────────────────────   ─────────────────────────────────────────
 N         (newest commit after Step 1 sets the property)  critical incidents are `critical`      ← known-good version
 N+1       UPDATE severity = 'low' WHERE 'critical'       the same rows now say `low`            ← Hydra (Step 2)
 N+2       RESTORE TABLE TO VERSION AS OF N               `critical` again; N+1 stays in history ← undo (Step 6)
```

| Tool | Answers | Step |
|---|---|---|
| `DESCRIBE HISTORY` | which commit, when, by whom, with which metrics | 3 and 7 |
| **Change data feed**: `table_changes('table', from, to)` | which rows changed, with their value before (`update_preimage`) and after (`update_postimage`) | 4 |
| **Time travel**: `... VERSION AS OF n` (or `TIMESTAMP AS OF`) | what did the table look like at version n? | 5 |
| `RESTORE TABLE ... TO VERSION AS OF n` | put the table back as it was at version n | 6 |

**How this notebook implements it**

* The table is `shield.silver.incident_reports`. Step 1 sets the table property `delta.enableChangeDataFeed = true` and remembers the newest version as `good_version`. Step 2 runs the tampering `UPDATE`, whose version is `tamper_version`.
* Steps 3 to 5 look at the damage three ways (commit log, row changes, old snapshot). Step 6 restores `good_version`; Step 7 shows that the tampering is still in the history.
* The change feed used here is the per-table property, sometimes called legacy change data feed. Databricks also offers an automatic change data feed (Databricks Runtime 19 or above, needs row tracking, no property) that you must not mix with it. It is not used here.

**Why it helps:** you get an audit trail, you can reproduce yesterday's report exactly, and you can undo a bad write in one statement. `RESTORE` is a new commit, so nothing is erased and the mistake stays visible.

**Limits you should know**

* Time travel only reaches back while the data files still exist. `VACUUM` removes them after the retention period (`delta.deletedFileRetentionDuration`, 7 days by default), and recent runtimes block time travel past that period. History itself is kept 30 days (`delta.logRetentionDuration`).
* The change feed only records changes made **after** it was switched on, and `VACUUM` deletes its files too.
* It is not a backup: a dropped table or deleted storage cannot be restored this way.

**Not covered yet:** what `VACUUM` does to time travel (a mini-project in the doc), and building a permanent audit table or slowly changing dimension from the feed.

## Setup

In [ ]:
def show_checks(checks):
    """Show (check, result, expected) rows with a ✅ / ❌ for whether result matches expected."""
    fmt = lambda v: f"{v:,}" if isinstance(v, int) and not isinstance(v, bool) else str(v)
    rows = [(what, fmt(got), fmt(want), "✅" if got == want else "❌") for what, got, want in checks]
    display(spark.createDataFrame(rows, "check string, result string, expected string, ok string"))


from pyspark.sql import functions as F

INCIDENTS_TABLE = "shield.silver.incident_reports"

assert spark.catalog.tableExists(INCIDENTS_TABLE), "Run notebook 06 first: shield.silver.incident_reports is missing."
assert spark.table(INCIDENTS_TABLE).filter("severity = 'critical'").count() > 0, (
    "shield.silver.incident_reports has no 'critical' incidents. Either chapter 06 has not run, or an earlier run of this notebook "
    "stopped after the tampering in Step 2. Re-run notebook 06 to rebuild the table, then start here again."
)

## 1 · Prepare

### Step 1 · Mark the known-good version and turn on the change feed
**Goal:** write down the version to come back to, and start recording row-level changes before anyone tampers.

| Input | Value |
|---|---|
| `shield.silver.incident_reports` as chapter 06 left it | 9,500 incidents |

The severity breakdown you should see now:

| severity | n |
|---|---|
| low | 4,722 |
| medium | 2,915 |
| high | 1,428 |
| critical | 435 |

**Expected:** the table property reads `true`, and `good_version` is a whole number: the newest version of the table. Its value depends on which chapters touched the table before, so it is not predictable and not checked.
The row count and the breakdown above hold for the table as chapter 06 builds it. Setting the property is safe to repeat.

In [ ]:
def severity_counts(table_expression: str) -> dict:
    """Rows per severity, for a table name or a table name with a time-travel clause."""
    rows = spark.sql(f"SELECT severity, COUNT(*) AS n FROM {table_expression} GROUP BY severity").collect()
    return {row["severity"]: row["n"] for row in rows}


def show_severities(counts: dict) -> None:
    display(spark.createDataFrame(list(counts.items()), "severity string, n long").orderBy(F.desc("n")))


spark.sql(f"ALTER TABLE {INCIDENTS_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
good_version = spark.sql(f"DESCRIBE HISTORY {INCIDENTS_TABLE}").agg(F.max("version")).first()[0]
print("Known-good version:", good_version)

table_properties = {row["key"]: row["value"] for row in spark.sql(f"SHOW TBLPROPERTIES {INCIDENTS_TABLE}").collect()}
severity_before = severity_counts(INCIDENTS_TABLE)
critical_before = severity_before["critical"]
show_severities(severity_before)

show_checks([
    ("change data feed property", table_properties.get("delta.enableChangeDataFeed"), "true"),
    ("incidents in the table", sum(severity_before.values()), 9_500),
    ("critical incidents", critical_before, 435),
])

## 2 · The tampering

### Step 2 · Hydra downgrades the critical incidents
**Goal:** make a damaging change we can then investigate and undo.

**Expected:** the `UPDATE` changes 435 rows (all the `critical` ones). Afterwards there are no `critical` incidents, `low` has 4,722 + 435 = 5,157, and the total is still 9,500.
The CHECK constraint from chapter 06 (`casualties >= 0`) does not object: it says nothing about severity.
The update is a new version, `tamper_version`. Run this cell once: a second run finds no `critical` rows, and Steps 3 and 4 could then point at an empty commit.

What you will see:

| severity | n |
|---|---|
| low | 5,157 |
| medium | 2,915 |
| high | 1,428 |

In [ ]:
spark.sql(f"UPDATE {INCIDENTS_TABLE} SET severity = 'low' WHERE severity = 'critical'")

severity_tampered = severity_counts(INCIDENTS_TABLE)
show_severities(severity_tampered)

tamper_version = (
    spark.sql(f"DESCRIBE HISTORY {INCIDENTS_TABLE}").filter("operation = 'UPDATE'").agg(F.max("version")).first()[0]
)
show_checks([
    ("critical incidents left", severity_tampered.get("critical", 0), 0),
    ("low incidents", severity_tampered["low"], severity_before["low"] + critical_before),
    ("incidents in the table", sum(severity_tampered.values()), sum(severity_before.values())),
    ("tamper version is after the known-good version", tamper_version > good_version, True),
])

## 3 · The investigation

### Step 3 · Who changed what, and when?
**Goal:** find the offending commit in the history without looking at any data.

**Expected:** the newest history row is the `UPDATE`, with your user name and `numUpdatedRows = 435` in its metrics. Below it, older rows from earlier chapters (and, on a first run, the commit that set the property).

What you will see (illustrative):

| version | timestamp | userName | operation | operationMetrics |
|---|---|---|---|---|
| N+1 | … | you@… | UPDATE | numUpdatedRows: 435, … |
| N | … | you@… | SET TBLPROPERTIES | … |
| … | … | … | … | … |

The cell only reads, so it is safe to re-run.

In [ ]:
incident_history = spark.sql(f"DESCRIBE HISTORY {INCIDENTS_TABLE}")
display(incident_history.select("version", "timestamp", "userName", "operation", "operationMetrics"))

update_commit = incident_history.filter(F.col("version") == tamper_version).first()
show_checks([
    ("operation of the tamper version", update_commit["operation"], "UPDATE"),
    ("rows updated by that commit", int(update_commit["operationMetrics"]["numUpdatedRows"]), critical_before),
    ("the commit records who ran it", update_commit["userName"] is not None, True),
])

### Step 4 · Which rows? The change feed
**Goal:** list the exact rows that changed, with their value before and after.

| Input | Value |
|---|---|
| `table_changes('shield.silver.incident_reports', tamper_version, tamper_version)` | the change feed of the single tampering commit (start and end are inclusive) |

**Expected:** 870 change rows: 435 `update_preimage` (severity `critical`, the old value) and 435 `update_postimage` (severity `low`, the new value). The sample shows five incidents with both their images, so 10 rows; it is a sample, which is why it is cut.
This works because the feed was switched on in Step 1, before the update. Asking for a version from before that raises an error.

What you will see (five incidents, illustrative ids):

| incident_id | severity | _change_type | _commit_version |
|---|---|---|---|
| I0000123 | low | update_postimage | N+1 |
| I0000123 | critical | update_preimage | N+1 |
| … | … | … | … |

The cell only reads, so it is safe to re-run.

In [ ]:
change_feed = spark.sql(f"SELECT * FROM table_changes('{INCIDENTS_TABLE}', {tamper_version}, {tamper_version})")
display(change_feed.groupBy("_change_type").count().orderBy("_change_type"))

sample_ids = [row["incident_id"] for row in change_feed.filter("_change_type = 'update_preimage'").limit(5).collect()]
display(
    change_feed.filter(F.col("incident_id").isin(sample_ids))
    .select("incident_id", "severity", "_change_type", "_commit_version")
    .orderBy("incident_id", "_change_type")
)

changes_by_type = {row["_change_type"]: row["count"] for row in change_feed.groupBy("_change_type").count().collect()}
severities_by_type = {
    row["_change_type"]: row["severity"] for row in change_feed.select("_change_type", "severity").distinct().collect()
}
show_checks([
    ("update_preimage rows", changes_by_type.get("update_preimage"), critical_before),
    ("update_postimage rows", changes_by_type.get("update_postimage"), critical_before),
    ("all change rows", sum(changes_by_type.values()), 2 * critical_before),
    ("severity before the update", severities_by_type.get("update_preimage"), "critical"),
    ("severity after the update", severities_by_type.get("update_postimage"), "low"),
])

### Step 5 · What did the table look like before? Time travel
**Goal:** read the old snapshot without changing the table.

| Input | Value |
|---|---|
| `shield.silver.incident_reports VERSION AS OF good_version` | the table before the tampering |

**Expected:** the breakdown from Step 1 again, with 435 `critical` incidents, while the current table still has none. Reading an old version needs the old data files, which Delta keeps for 7 days by default, so the snapshot stays readable for that long.
Use `TIMESTAMP AS OF '2025-…'` instead of a version when you know the time and not the number (a mini-project in the doc). The cell only reads, so it is safe to re-run.

In [ ]:
severity_at_good_version = severity_counts(f"{INCIDENTS_TABLE} VERSION AS OF {good_version}")
show_severities(severity_at_good_version)

show_checks([
    ("critical incidents at the known-good version", severity_at_good_version["critical"], critical_before),
    ("critical incidents in the current table", severity_counts(INCIDENTS_TABLE).get("critical", 0), 0),
    ("snapshot equals the Step 1 breakdown", severity_at_good_version == severity_before, True),
])

## 4 · The repair

### Step 6 · Undo it: RESTORE
**Goal:** put the table back to the known-good version.

**Expected:** the breakdown is identical to Step 1 (435 `critical` again). `RESTORE` returns one row of metrics (files removed and restored). It is a new commit, so the newest history row is `RESTORE`.
Running it again restores the same version once more: harmless, but it adds another `RESTORE` commit.

What you will see:

| severity | n |
|---|---|
| low | 4,722 |
| medium | 2,915 |
| high | 1,428 |
| critical | 435 |

In [ ]:
display(spark.sql(f"RESTORE TABLE {INCIDENTS_TABLE} TO VERSION AS OF {good_version}"))

severity_restored = severity_counts(INCIDENTS_TABLE)
show_severities(severity_restored)

latest_operation = spark.sql(f"DESCRIBE HISTORY {INCIDENTS_TABLE}").orderBy(F.desc("version")).first()["operation"]
show_checks([
    ("breakdown equals the Step 1 breakdown", severity_restored == severity_before, True),
    ("critical incidents", severity_restored["critical"], critical_before),
    ("newest operation in the history", latest_operation, "RESTORE"),
])

### Step 7 · The tampering stays in the history
**Goal:** confirm that nothing was erased: the undo is itself a commit.

**Expected:** the history now has `RESTORE` on top of `UPDATE`, and the `UPDATE` is still readable (Step 4 would still work). The display shows only the newest five rows, because older rows belong to earlier chapters.
On a first run the next row down is the commit that set the property.

What you will see (illustrative):

| version | operation | userName |
|---|---|---|
| N+2 | RESTORE | you@… |
| N+1 | UPDATE | you@… |
| N | SET TBLPROPERTIES | you@… |
| … | … | … |

The cell only reads, so it is safe to re-run.

In [ ]:
final_history = spark.sql(f"DESCRIBE HISTORY {INCIDENTS_TABLE}")
display(final_history.select("version", "operation", "userName").orderBy(F.desc("version")).limit(5))

newest_two = [row["operation"] for row in final_history.orderBy(F.desc("version")).limit(2).collect()]
restore_version = final_history.filter("operation = 'RESTORE'").agg(F.max("version")).first()[0]
show_checks([
    ("two newest operations", newest_two, ["RESTORE", "UPDATE"]),
    ("the tampering commit is still listed", final_history.filter(F.col("version") == tamper_version).count(), 1),
    ("restore came after the tampering", restore_version > tamper_version, True),
])

## 👀 In the workspace

* **Catalog** → `shield` → `silver` → `incident_reports` → **History** tab: the `UPDATE` and the `RESTORE`, with the user and this notebook.
* The **Details** tab (or `SHOW TBLPROPERTIES shield.silver.incident_reports`) shows `delta.enableChangeDataFeed` until the clean-up switches it off.

## 🧹 Clean up

The data is already back to how it was. The cell below switches the change feed off again so the table keeps its original properties; it does not remove the extra versions, because history cannot be deleted (it ages out after 30 days).
Skip it if you want to keep recording row-level changes. Chapters 10 and 11 work either way.

In [ ]:
spark.sql(f"ALTER TABLE {INCIDENTS_TABLE} UNSET TBLPROPERTIES IF EXISTS ('delta.enableChangeDataFeed')")

## 🎯 Recap

* Every commit is a version: `DESCRIBE HISTORY` shows who changed the table and when, with row counts in the metrics.
* The change feed (`table_changes`) shows the rows themselves, before and after, but only for changes made after you turned it on.
* Time travel reads an old version without touching the table; `RESTORE` turns the table back and is itself a new version, so the evidence stays.
* None of this is a backup, and `VACUUM` shortens how far back you can go.

Mini-projects: [docs/07-doctor-strange-time-stone.md](../../docs/07-doctor-strange-time-stone.md#try-it-yourself)

> 💡 Next: chapter 08 rebuilds bronze, silver and gold as a declarative pipeline that handles quality and increments for you.